Futures prices exhibit medium-term persistence: instruments with positive returns over the preceding several weeks tend to continue rising, while instruments with negative returns tend to continue falling

In [139]:
from src.data_loader import prepare_futures_data
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
from src.data_loader import prepare_futures_data
from src.bar_builder import build_15min_bars
from src.signal import add_high_volume_breakout_signal

In [140]:
if_df = prepare_futures_data("data/IF9999.CCFX.csv")
if_df['day'] = if_df['date'].dt.date
if_df.drop(['is_valid_1m_pair', 'clean_return_1m'], axis=1, inplace=True)
print(if_df.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 09:16:00  3450.0  3488.0  3450.0  3470.0   490.0  509212140.0   
1 2010-04-16 09:17:00  3468.0  3473.8  3467.0  3467.0   306.0  318622800.0   
2 2010-04-16 09:18:00  3467.0  3471.0  3466.0  3467.0   198.0  206012040.0   
3 2010-04-16 09:19:00  3467.0  3468.2  3448.0  3448.0   282.0  292809960.0   
4 2010-04-16 09:20:00  3448.0  3460.0  3448.0  3454.0   248.0  257066520.0   

   open_interest  symbol  session_block         day  
0          359.0  IF1005              0  2010-04-16  
1          521.0  IF1005              0  2010-04-16  
2          630.0  IF1005              0  2010-04-16  
3          764.0  IF1005              0  2010-04-16  
4          795.0  IF1005              0  2010-04-16  


In [141]:
if_df_daily = if_df[(if_df['date'].dt.time >= dt.time(hour=9)) & (if_df['date'].dt.time <= dt.time(hour=15))].copy()
if_df_daily = if_df_daily.sort_values('date').groupby('day').tail(1)
if_df_daily.reset_index(inplace=True, drop=True)
print('Duplicate:', if_df_daily['day'].duplicated().sum())
print(if_df_daily.head(1))

Duplicate: 0
                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   

   open_interest  symbol  session_block         day  
0         2989.0  IF1005              1  2010-04-16  


In [142]:
print(if_df_daily.tail(1))

                    date    open    high     low   close  volume  \
2794 2021-10-18 15:00:00  4860.8  4861.6  4857.8  4860.0   779.0   

             money  open_interest  symbol  session_block         day  
2794  1.135937e+09        71872.0  IF2111           5657  2021-10-18  


In [143]:
print(if_df_daily['close'].isna().sum())

0


In [144]:
print(len(if_df_daily['symbol'].unique()))

139


In [145]:
if_df_daily['prev_symbol'] = if_df_daily['symbol'].shift(1)
if_df_daily['prev_close'] = if_df_daily['close'].shift(1)
if_df_daily['symbol_change'] = if_df_daily['prev_symbol'] != if_df_daily['symbol']
if_df_daily.loc[if_df_daily.index[0], "symbol_change"] = False
if_df_daily['raw_daily_return'] = if_df_daily['close'] / if_df_daily['prev_close'] - 1
print(if_df_daily.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   
1 2010-04-19 15:00:00  3201.8  3203.0  3197.0  3198.4   464.0  445485780.0   
2 2010-04-20 15:00:00  3216.0  3217.2  3215.8  3216.8   204.0  196855560.0   
3 2010-04-21 15:00:00  3271.4  3272.2  3267.0  3269.0   791.0  775723260.0   
4 2010-04-22 15:00:00  3234.4  3236.0  3234.0  3234.4   175.0  169837320.0   

   open_interest  symbol  session_block         day prev_symbol  prev_close  \
0         2989.0  IF1005              1  2010-04-16        None         NaN   
1         5060.0  IF1005              3  2010-04-19      IF1005      3426.0   
2         5632.0  IF1005              5  2010-04-20      IF1005      3198.4   
3         6153.0  IF1005              7  2010-04-21      IF1005      3216.8   
4         6414.0  IF1005              9  2010-04-22      IF1005      3269.0   

   symbol_change  raw_daily_return  
0          False   

In [146]:
if_df_daily_change_symbol_statistic = if_df_daily.groupby('symbol_change').agg(
    count=('day', 'size'),
    mean_return=('raw_daily_return', 'mean'),
    median_return=('raw_daily_return', 'median'),
    std_return=('raw_daily_return', 'std'),
    max_return=('raw_daily_return', 'max'),
    min_return=('raw_daily_return', 'min'),
)
print(if_df_daily_change_symbol_statistic)

               count  mean_return  median_return  std_return  max_return  \
symbol_change                                                              
False           2657     0.000328       0.000066    0.015754    0.100075   
True             138    -0.001199      -0.000087    0.018722    0.057976   

               min_return  
symbol_change              
False           -0.100896  
True            -0.081392  


In [147]:
if_df_daily['clean_daily_return'] = np.where(if_df_daily['symbol_change'],
                                       0,
                                       if_df_daily['raw_daily_return'],
)
if_df_daily.loc[if_df_daily.index[0], "clean_daily_return"] = 0
print(if_df_daily.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   
1 2010-04-19 15:00:00  3201.8  3203.0  3197.0  3198.4   464.0  445485780.0   
2 2010-04-20 15:00:00  3216.0  3217.2  3215.8  3216.8   204.0  196855560.0   
3 2010-04-21 15:00:00  3271.4  3272.2  3267.0  3269.0   791.0  775723260.0   
4 2010-04-22 15:00:00  3234.4  3236.0  3234.0  3234.4   175.0  169837320.0   

   open_interest  symbol  session_block         day prev_symbol  prev_close  \
0         2989.0  IF1005              1  2010-04-16        None         NaN   
1         5060.0  IF1005              3  2010-04-19      IF1005      3426.0   
2         5632.0  IF1005              5  2010-04-20      IF1005      3198.4   
3         6153.0  IF1005              7  2010-04-21      IF1005      3216.8   
4         6414.0  IF1005              9  2010-04-22      IF1005      3269.0   

   symbol_change  raw_daily_return  clean_daily_return  

In [148]:
print(if_df_daily['clean_daily_return'].isna().sum())
print(if_df_daily[if_df_daily['symbol_change'] == True].head())

0
                   date    open    high     low   close  volume        money  \
22  2010-05-19 15:00:00  2789.8  2792.6  2789.6  2791.8   271.0  226894260.0   
41  2010-06-18 15:00:00  2713.8  2714.4  2710.6  2712.8   948.0  771429420.0   
61  2010-07-16 15:00:00  2618.2  2618.6  2617.6  2618.2   546.0  428841000.0   
85  2010-08-19 15:00:00  2978.0  2978.2  2976.8  2976.8   625.0  558274920.0   
106 2010-09-17 15:00:00  2875.0  2875.2  2874.6  2875.0   202.0  174225240.0   

     open_interest  symbol  session_block         day prev_symbol  prev_close  \
22         14648.0  IF1006             46  2010-05-19      IF1005      2765.8   
41         22489.0  IF1007             85  2010-06-18      IF1006      2748.4   
61         24259.0  IF1008            126  2010-07-16      IF1007      2608.6   
85         30733.0  IF1009            175  2010-08-19      IF1008      2946.0   
106        25071.0  IF1010            218  2010-09-17      IF1009      2860.4   

     symbol_change  raw_daily_

In [149]:
normal_rows = if_df_daily.loc[
    ~if_df_daily["symbol_change"]
]

return_difference = (
    normal_rows["raw_daily_return"]
    - normal_rows["clean_daily_return"]
)

max_abs_difference = return_difference.abs().max()

print(max_abs_difference)

0.0


In [150]:
if_df_daily["continuous_price"] = (
    100
    * (1 + if_df_daily["clean_daily_return"]).cumprod()
)
print(if_df_daily.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   
1 2010-04-19 15:00:00  3201.8  3203.0  3197.0  3198.4   464.0  445485780.0   
2 2010-04-20 15:00:00  3216.0  3217.2  3215.8  3216.8   204.0  196855560.0   
3 2010-04-21 15:00:00  3271.4  3272.2  3267.0  3269.0   791.0  775723260.0   
4 2010-04-22 15:00:00  3234.4  3236.0  3234.0  3234.4   175.0  169837320.0   

   open_interest  symbol  session_block         day prev_symbol  prev_close  \
0         2989.0  IF1005              1  2010-04-16        None         NaN   
1         5060.0  IF1005              3  2010-04-19      IF1005      3426.0   
2         5632.0  IF1005              5  2010-04-20      IF1005      3198.4   
3         6153.0  IF1005              7  2010-04-21      IF1005      3216.8   
4         6414.0  IF1005              9  2010-04-22      IF1005      3269.0   

   symbol_change  raw_daily_return  clean_daily_return  

In [151]:
first_change_index = if_df_daily.loc[
    if_df_daily["symbol_change"]
].index[0]


print(if_df_daily.loc[first_change_index - 1:first_change_index + 1])

                  date    open    high     low   close  volume        money  \
21 2010-05-18 15:00:00  2767.2  2767.4  2764.2  2765.8    81.0   67213920.0   
22 2010-05-19 15:00:00  2789.8  2792.6  2789.6  2791.8   271.0  226894260.0   
23 2010-05-20 15:00:00  2751.8  2753.2  2750.8  2752.6   486.0  401298120.0   

    open_interest  symbol  session_block         day prev_symbol  prev_close  \
21         2240.0  IF1005             43  2010-05-18      IF1005      2712.4   
22        14648.0  IF1006             46  2010-05-19      IF1005      2765.8   
23        16964.0  IF1006             48  2010-05-20      IF1006      2791.8   

    symbol_change  raw_daily_return  clean_daily_return  continuous_price  
21          False          0.019687            0.019687         80.729714  
22           True          0.009401            0.000000         80.729714  
23          False         -0.014041           -0.014041         79.596178  


In [152]:
contin_price_20d = if_df_daily['continuous_price'].shift(20) 
future_contin_price_5d = if_df_daily['continuous_price'].shift(-5)
if_df_daily['past_return_20d'] = if_df_daily['continuous_price'] / contin_price_20d - 1
if_df_daily['future_return_5d'] = future_contin_price_5d / if_df_daily['continuous_price'] - 1
print(if_df_daily.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   
1 2010-04-19 15:00:00  3201.8  3203.0  3197.0  3198.4   464.0  445485780.0   
2 2010-04-20 15:00:00  3216.0  3217.2  3215.8  3216.8   204.0  196855560.0   
3 2010-04-21 15:00:00  3271.4  3272.2  3267.0  3269.0   791.0  775723260.0   
4 2010-04-22 15:00:00  3234.4  3236.0  3234.0  3234.4   175.0  169837320.0   

   open_interest  symbol  session_block         day prev_symbol  prev_close  \
0         2989.0  IF1005              1  2010-04-16        None         NaN   
1         5060.0  IF1005              3  2010-04-19      IF1005      3426.0   
2         5632.0  IF1005              5  2010-04-20      IF1005      3198.4   
3         6153.0  IF1005              7  2010-04-21      IF1005      3216.8   
4         6414.0  IF1005              9  2010-04-22      IF1005      3269.0   

   symbol_change  raw_daily_return  clean_daily_return  

In [153]:
start_date_train = dt.datetime(year=2015, month=1, day=1)
end_date_train = dt.datetime(year=2018, month=12, day=31, hour=23, minute=59, second=59)
if_df_daily_train = if_df_daily[(if_df_daily['date'] >= start_date_train) 
                                & (if_df_daily['date'] <= end_date_train)
                                & (if_df_daily['future_return_5d'].notna())].copy()
print(if_df_daily_train['future_return_5d'].corr(if_df_daily_train['past_return_20d']))

0.060186255155930346


In [154]:
if_df_daily_train["return_quintile"] = pd.qcut(
    if_df_daily_train["past_return_20d"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)

if_df_daily_qcut_statistic = if_df_daily_train.groupby('return_quintile', observed=False).agg(
    count=('date', 'size'),
    min_past_return_20d=('past_return_20d', 'min'),
    max_past_return_20d=('past_return_20d', 'max'),
    mean_future_return_5d=('future_return_5d', 'mean'),
    median_future_return_5d=('future_return_5d', 'median'),
)
print(if_df_daily_qcut_statistic)

                 count  min_past_return_20d  max_past_return_20d  \
return_quintile                                                    
Q1                 195            -0.300815            -0.040410   
Q2                 195            -0.040290            -0.003900   
Q3                 195            -0.003791             0.025913   
Q4                 195             0.026235             0.052652   
Q5                 195             0.052662             0.201102   

                 mean_future_return_5d  median_future_return_5d  
return_quintile                                                  
Q1                            0.000274                 0.002882  
Q2                           -0.004323                -0.000245  
Q3                            0.001682                 0.005171  
Q4                            0.002806                 0.003233  
Q5                            0.005941                 0.005718  


In [155]:
if_df_daily["future_date_5d"] = if_df_daily["date"].shift(-5)

if_df_daily_train = if_df_daily.loc[
    (if_df_daily["date"] >= start_date_train)
    & (if_df_daily["date"] <= end_date_train)
    & (if_df_daily["future_date_5d"] <= end_date_train)
    & if_df_daily["past_return_20d"].notna()
    & if_df_daily["future_return_5d"].notna()
].copy()

threshold_80 = if_df_daily_train["past_return_20d"].quantile(0.80)

if_df_daily_train_q5 = if_df_daily_train.loc[
    if_df_daily_train["past_return_20d"] >= threshold_80
].copy()

if_df_daily_train_q5["year"] = (
    if_df_daily_train_q5["date"].dt.year
)

yearly_q5_summary = (
    if_df_daily_train_q5
    .groupby("year")
    .agg(
        signals=("future_return_5d", "size"),
        mean_future_return_5d=("future_return_5d", "mean"),
        median_future_return_5d=("future_return_5d", "median"),
        win_rate=("future_return_5d", lambda x: (x > 0).mean()),
    )
)

yearly_q5_summary["mean_future_return_5d_pct"] = (
    yearly_q5_summary["mean_future_return_5d"] * 100
)

yearly_q5_summary["median_future_return_5d_pct"] = (
    yearly_q5_summary["median_future_return_5d"] * 100
)

yearly_q5_summary["win_rate_pct"] = (
    yearly_q5_summary["win_rate"] * 100
)

print("Train threshold:", threshold_80)
print("Train threshold, %:", threshold_80 * 100)

print(
    yearly_q5_summary[
        [
            "signals",
            "mean_future_return_5d_pct",
            "median_future_return_5d_pct",
            "win_rate_pct",
        ]
    ]
)

Train threshold: 0.05267246314843175
Train threshold, %: 5.267246314843175
      signals  mean_future_return_5d_pct  median_future_return_5d_pct  \
year                                                                    
2015      115                   1.184925                     1.488955   
2016       42                  -0.266676                    -0.760890   
2017       23                  -0.365364                    -0.422775   
2018       14                  -0.512765                     0.791479   

      win_rate_pct  
year                
2015     66.086957  
2016     42.857143  
2017     39.130435  
2018     57.142857  


In [156]:
signal_candidates = (
    if_df_daily_train_q5
    .sort_values("date")
    .copy()
)

selected_indices = []
last_exit_date = None

for index, row in signal_candidates.iterrows():
    entry_date = row["date"]
    exit_date = row["future_date_5d"]

    if (
        last_exit_date is None
        or entry_date > last_exit_date
        ):
        selected_indices.append(index)
        last_exit_date = exit_date

non_overlapping_trades = (
    signal_candidates
    .loc[selected_indices]
    .copy()
)

non_overlapping_trades["year"] = (
    non_overlapping_trades["date"].dt.year
)

print("Signals before:", len(signal_candidates))
print("Independent trades:", len(non_overlapping_trades))

Signals before: 194
Independent trades: 51


In [157]:
yearly_non_overlapping_summary = (
    non_overlapping_trades
    .groupby("year")
    .agg(
        trades=("future_return_5d", "size"),
        mean_return=("future_return_5d", "mean"),
        median_return=("future_return_5d", "median"),
        win_rate=("future_return_5d", lambda x: (x > 0).mean()),
    )
)

yearly_non_overlapping_summary["mean_return_pct"] = (
    yearly_non_overlapping_summary["mean_return"] * 100
)

yearly_non_overlapping_summary["median_return_pct"] = (
    yearly_non_overlapping_summary["median_return"] * 100
)

yearly_non_overlapping_summary["win_rate_pct"] = (
    yearly_non_overlapping_summary["win_rate"] * 100
)

print(
    yearly_non_overlapping_summary[
        [
            "trades",
            "mean_return_pct",
            "median_return_pct",
            "win_rate_pct",
        ]
    ]
)

      trades  mean_return_pct  median_return_pct  win_rate_pct
year                                                          
2015      26         0.844254           1.412053     65.384615
2016      12         0.981754           0.655231     75.000000
2017       9        -0.344156          -0.028305     44.444444
2018       4        -1.303668          -0.706610     50.000000


In [158]:
contin_price_60d = if_df_daily['continuous_price'].shift(60)
future_price_20d = if_df_daily['continuous_price'].shift(-20)
if_df_daily['past_return_60d'] = if_df_daily['continuous_price'] / contin_price_60d - 1
if_df_daily['future_return_20d'] = future_price_20d / if_df_daily['continuous_price'] - 1
print(if_df_daily.head())

                 date    open    high     low   close  volume        money  \
0 2010-04-16 15:00:00  3428.2  3428.2  3425.2  3426.0   253.0  260067720.0   
1 2010-04-19 15:00:00  3201.8  3203.0  3197.0  3198.4   464.0  445485780.0   
2 2010-04-20 15:00:00  3216.0  3217.2  3215.8  3216.8   204.0  196855560.0   
3 2010-04-21 15:00:00  3271.4  3272.2  3267.0  3269.0   791.0  775723260.0   
4 2010-04-22 15:00:00  3234.4  3236.0  3234.0  3234.4   175.0  169837320.0   

   open_interest  symbol  session_block  ... prev_close symbol_change  \
0         2989.0  IF1005              1  ...        NaN         False   
1         5060.0  IF1005              3  ...     3426.0         False   
2         5632.0  IF1005              5  ...     3198.4         False   
3         6153.0  IF1005              7  ...     3216.8         False   
4         6414.0  IF1005              9  ...     3269.0         False   

   raw_daily_return  clean_daily_return  continuous_price  past_return_20d  \
0             

In [159]:
if_df_daily["future_date_20d"] = if_df_daily["date"].shift(-20)

if_df_daily_train = if_df_daily.loc[
    (if_df_daily["date"] >= start_date_train)
    & (if_df_daily["date"] <= end_date_train)
    & (if_df_daily["future_date_20d"] <= end_date_train)
    & if_df_daily["past_return_60d"].notna()
    & if_df_daily["future_return_20d"].notna()
].copy()
print(if_df_daily_train.head())

                    date    open    high     low   close   volume  \
1144 2015-01-05 15:00:00  3683.0  3686.0  3675.0  3675.8  11358.0   
1145 2015-01-06 15:00:00  3660.0  3661.0  3652.4  3654.4   4907.0   
1146 2015-01-07 15:00:00  3653.0  3654.8  3652.2  3653.8   2703.0   
1147 2015-01-08 15:00:00  3564.6  3568.4  3563.6  3567.0   3199.0   
1148 2015-01-09 15:00:00  3521.4  3531.4  3518.0  3525.6   6856.0   

             money  open_interest  symbol  session_block  ... symbol_change  \
1144  1.254311e+10       148547.0  IF1501           2345  ...         False   
1145  5.381269e+09       135585.0  IF1501           2347  ...         False   
1146  2.962573e+09       125532.0  IF1501           2349  ...         False   
1147  3.422517e+09       115018.0  IF1501           2351  ...         False   
1148  7.250060e+09        93374.0  IF1501           2353  ...         False   

     raw_daily_return  clean_daily_return  continuous_price  past_return_20d  \
1144         0.028771         

In [161]:
print(if_df_daily_train['future_return_20d'].corr(if_df_daily_train['past_return_60d']))

-0.1425118307031485


In [173]:
if_df_daily_train["return_quintile"] = pd.qcut(
    if_df_daily_train["past_return_60d"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)

if_df_daily_qcut_statistic = if_df_daily_train.groupby('return_quintile', observed=False).agg(
    count=('date', 'size'),
    min_past_return_60d=('past_return_60d', 'min'),
    max_past_return_60d=('past_return_60d', 'max'),
    mean_future_return_20d=('future_return_20d', 'mean'),
    median_future_return_20d=('future_return_20d', 'median'),
    win_rate=("future_return_20d", lambda x: (x > 0).mean()),
)
print(if_df_daily_qcut_statistic)

                 count  min_past_return_60d  max_past_return_60d  \
return_quintile                                                    
Q1                 191            -0.408984            -0.084923   
Q2                 191            -0.084867             0.016270   
Q3                 191             0.016585             0.064533   
Q4                 191             0.064846             0.103750   
Q5                 191             0.103982             0.504636   

                 mean_future_return_20d  median_future_return_20d  win_rate  
return_quintile                                                              
Q1                             0.009709                  0.002263  0.518325  
Q2                            -0.008598                 -0.011857  0.439791  
Q3                             0.025212                  0.029356  0.806283  
Q4                             0.013751                  0.015521  0.623037  
Q5                            -0.008733                

In [175]:
if_df_daily_train = if_df_daily_train.copy()
if_df_daily_train["year"] = if_df_daily_train["date"].dt.year

yearly_quintile_summary = (
    if_df_daily_train
    .groupby(["year", "return_quintile"], observed=True)
    .agg(
        observations=("future_return_20d", "size"),
        mean_future_return_20d=("future_return_20d", "mean"),
        median_future_return_20d=("future_return_20d", "median"),
        positive_return_rate=(
            "future_return_20d",
            lambda x: (x > 0).mean(),
        ),
    )
)

yearly_quintile_summary["mean_future_return_20d_pct"] = (
    yearly_quintile_summary["mean_future_return_20d"] * 100
)

yearly_quintile_summary["median_future_return_20d_pct"] = (
    yearly_quintile_summary["median_future_return_20d"] * 100
)

yearly_quintile_summary["positive_return_rate_pct"] = (
    yearly_quintile_summary["positive_return_rate"] * 100
)

print(
    yearly_quintile_summary.loc[
        (
            slice(None),
            ["Q1", "Q3", "Q5"],
        ),
        [
            "observations",
            "mean_future_return_20d_pct",
            "median_future_return_20d_pct",
            "positive_return_rate_pct",
        ],
    ]
)

                      observations  mean_future_return_20d_pct  \
year return_quintile                                             
2015 Q1                         51                    2.527408   
     Q3                         10                    6.490937   
     Q5                        138                   -0.880688   
2016 Q1                         41                    4.787842   
     Q3                         75                    1.541419   
     Q5                         24                   -2.832619   
2017 Q3                         97                    3.582541   
     Q5                         29                    0.783276   
2018 Q1                         99                   -1.411662   
     Q3                          9                   -5.164567   

                      median_future_return_20d_pct  positive_return_rate_pct  
year return_quintile                                                          
2015 Q1                                   4.75512

In [180]:
if_df_daily_train['past_return_60d_abs'] = if_df_daily_train['past_return_60d'].abs()
if_df_daily_train['future_return_20d_abs'] = if_df_daily_train['future_return_20d'].abs()
print(if_df_daily_train['past_return_60d_abs'].corr(if_df_daily_train['future_return_20d_abs']))

0.3921439409345438


In [181]:
spearman_correlation = if_df_daily_train["past_return_60d_abs"].corr(
    if_df_daily_train["future_return_20d_abs"],
    method="spearman",
)

print(spearman_correlation)

0.16704126645558193


In [182]:
if_df_daily_train["return_quintile"] = pd.qcut(
    if_df_daily_train["past_return_60d_abs"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)

if_df_daily_qcut_statistic = if_df_daily_train.groupby('return_quintile', observed=False).agg(
    count=('date', 'size'),
    min_past_return_60d_abs=('past_return_60d_abs', 'min'),
    max_past_return_60d_abs=('past_return_60d_abs', 'max'),
    mean_future_return_20d_abs=('future_return_20d_abs', 'mean'),
    median_future_return_20d_abs=('future_return_20d_abs', 'median'),
)
print(if_df_daily_qcut_statistic)

                 count  min_past_return_60d_abs  max_past_return_60d_abs  \
return_quintile                                                            
Q1                 191                 0.000414                 0.041836   
Q2                 191                 0.041932                 0.071975   
Q3                 191                 0.072120                 0.096014   
Q4                 191                 0.096549                 0.145183   
Q5                 191                 0.146148                 0.504636   

                 mean_future_return_20d_abs  median_future_return_20d_abs  
return_quintile                                                            
Q1                                 0.047505                      0.038339  
Q2                                 0.040626                      0.039389  
Q3                                 0.042040                      0.032047  
Q4                                 0.043675                      0.029541  
Q5         

In [183]:
volatility_regime_df = if_df_daily_train.copy()

volatility_regime_df["year"] = (
    volatility_regime_df["date"].dt.year
)

volatility_regime_df["volatility_regime"] = np.where(
    volatility_regime_df["return_quintile"] == "Q5",
    "Q5",
    "Q1-Q4",
)

yearly_volatility_regime = (
    volatility_regime_df
    .groupby(
        ["year", "volatility_regime"],
        observed=True,
    )
    .agg(
        observations=("future_return_20d_abs", "size"),
        mean_future_abs_return=(
            "future_return_20d_abs",
            "mean",
        ),
        median_future_abs_return=(
            "future_return_20d_abs",
            "median",
        ),
    )
    .reset_index()
)

yearly_volatility_regime["mean_future_abs_return_pct"] = (
    yearly_volatility_regime["mean_future_abs_return"] * 100
)

yearly_volatility_regime["median_future_abs_return_pct"] = (
    yearly_volatility_regime["median_future_abs_return"] * 100
)

print(
    yearly_volatility_regime[
        [
            "year",
            "volatility_regime",
            "observations",
            "mean_future_abs_return_pct",
            "median_future_abs_return_pct",
        ]
    ]
)

   year volatility_regime  observations  mean_future_abs_return_pct  \
0  2015             Q1-Q4            82                   10.344498   
1  2015                Q5           162                   10.010821   
2  2016             Q1-Q4           233                    3.459698   
3  2016                Q5            11                    7.075338   
4  2017             Q1-Q4           244                    3.127900   
5  2018             Q1-Q4           205                    4.404371   
6  2018                Q5            18                    1.612857   

   median_future_abs_return_pct  
0                     10.336517  
1                      9.069630  
2                      3.217175  
3                      6.735459  
4                      3.031193  
5                      3.560749  
6                      1.266490  


In [184]:
median_pivot = yearly_volatility_regime.pivot(
    index="year",
    columns="volatility_regime",
    values="median_future_abs_return",
)

median_pivot["q5_to_normal_ratio"] = (
    median_pivot["Q5"] / median_pivot["Q1-Q4"]
)

print(median_pivot)

volatility_regime     Q1-Q4        Q5  q5_to_normal_ratio
year                                                     
2015               0.103365  0.090696            0.877436
2016               0.032172  0.067355            2.093594
2017               0.030312       NaN                 NaN
2018               0.035607  0.012665            0.355681


Absolute 60-day return does not provide a stable standalone forecast of subsequent 20-day volatility. The apparent full-sample relationship is largely explained by the high-volatility 2015 regime and does not persist consistently across years